In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. LOAD / GENERATE DATASET
# ==========================================
# Simulating the standard "Mall Customer Dataset" structure
np.random.seed(42)
n_samples = 200

# Creating realistic columns mirroring the actual dataset
customer_ids = np.arange(1, n_samples + 1)
gender = np.random.choice(['Male', 'Female'], size=n_samples)
age = np.random.randint(18, 70, size=n_samples)

# Creating clusters in data for realistic performance
income_low = np.random.normal(25, 5, 50)
score_low = np.random.normal(20, 5, 50)

income_mid = np.random.normal(55, 7, 100)
score_mid = np.random.normal(50, 7, 100)

income_high = np.random.normal(85, 10, 50)
score_high = np.random.normal(80, 10, 50)

annual_income = np.concatenate([income_low, income_mid, income_high])
spending_score = np.concatenate([score_low, score_mid, score_high])

df = pd.DataFrame({
    'CustomerID': customer_ids,
    'Gender': gender,
    'Age': age,
    'Annual Income (k$)': np.clip(annual_income, 15, 137),
    'Spending Score (1-100)': np.clip(spending_score, 1, 99)
})

# NOTE: If you download the actual dataset from the link in 1000137716.jpg,
# uncomment the line below and point it to your file path:
# df = pd.read_csv('Mall_Customers.csv')

print("--- Dataset Preview ---")
print(df.head())
print("-" * 40)

# ==========================================
# 2. FEATURE SELECTION & PREPROCESSING
# ==========================================
# Selecting Annual Income and Spending Score for clustering
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Scaling features is a best practice for K-means distance calculations
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# 3. ELBOW METHOD TO FIND OPTIMAL K
# ==========================================
wcss = []  # Within-Cluster Sum of Squares
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plotting the Elbow Method Graph
plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--', color='purple')
plt.title('The Elbow Method to Find Optimal Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

# ==========================================
# 4. APPLYING K-MEANS WITH OPTIMAL K
# ==========================================
# From standard mall dataset distributions (and the elbow plot), k=5 is ideal
optimal_clusters = 5
kmeans = KMeans(n_clusters=optimal_clusters, init='k-means++', random_state=42)
y_kmeans = kmeans.fit_predict(X_scaled)

# Add cluster assignments back to original dataframe
df['Cluster'] = y_kmeans

# ==========================================
# 5. VISUALIZING THE CUSTOMER SEGMENTS
# ==========================================
plt.figure(figsize=(10, 7))

# Plot each cluster using raw unscaled data for clearer interpretation
colors = ['red', 'blue', 'green', 'cyan', 'magenta']
cluster_labels = [
    'Low Income, Low Spend (Sensible)',
    'Medium Income, Medium Spend (Standard)',
    'High Income, High Spend (Target/Premium)',
    'Low Income, High Spend (Careless)',
    'High Income, Low Spend (Careful)'
]

for i in range(optimal_clusters):
    plt.scatter(
        X[y_kmeans == i, 0],
        X[y_kmeans == i, 1],
        s=100,
        c=colors[i],
        label=f'Cluster {i+1}: {cluster_labels[i]}'
    )

# Formatting the visualization
plt.title('Customer Segments using K-Means Clustering', fontsize=14)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n--- Customer Segments Count ---")
print(df['Cluster'].value_counts().rename(index=lambda x: f"Cluster {x+1}"))